In [2]:
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

In [2]:
# Create new merged csv with time_since_last_timestamp
# Around 3-5 min execution time with SSD

G2AAS_december = "Dataset/G2AAS_performance_december.csv"
G2AAS_november = "Dataset/G2AAS_performance_november.csv"

nov_df = pd.read_csv(G2AAS_november)
dec_df = pd.read_csv(G2AAS_december)
print("Files read, step 1/4")

nov_df['timestamp'] = pd.to_datetime(nov_df['timestamp'], utc=True, format='ISO8601')
dec_df['timestamp'] = pd.to_datetime(dec_df['timestamp'], utc=True, format='ISO8601')

merged_df = pd.concat([nov_df, dec_df], ignore_index=True)
print("Files merged, step 2/4")

merged_df['time_since_last_timestamp'] = merged_df['timestamp'].diff().dt.total_seconds()
# Remove the first row since time_since cannot be computed
merged_df = merged_df.iloc[1:]
print("Timestamps diff calculated, step 3/4")

merged_df.to_csv("Dataset/mergedTimeStampDiff.csv", index=False)
print("File created, step 4/4")

File created, step 4/4


In [3]:
# Create sliding windows

df = pd.read_csv("Dataset/mergedTimeStampDiff.csv")

# !!! WRITE ABOUT THIS
# Remove HUGE outlier
# !!! WRITE ABOUT THIS
df = df[df['time_since_last_timestamp'] != 71407.876935]
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, format='ISO8601')

features = df.to_numpy()
timestamps = df['timestamp'].to_numpy()

seq_len = 50
stride = 5

# Create sliding windows size = seq_len, step = stride
windows = sliding_window_view(features, window_shape=seq_len, axis=0)[::stride, :]

start_timestamp = windows[:, 0, 0]
end_timestamp = windows[:, 0, -1]

windows_duration = windows[:, 2, :]
windows_ts = windows[:, 3, :]

windows_duration = windows_duration.astype(float)
windows_ts = windows_ts.astype(float)

# Compute moving averages, std
moving_avg_duration = windows_duration.mean(axis=1)
moving_avg_ts = windows_ts.mean(axis=1)
moving_std_duration = windows_duration.std(axis=1)
moving_std_ts = windows_ts.std(axis=1)

# Build output dataframe
window_df = pd.DataFrame({
    "start_timestamp": start_timestamp,
    "end_timestamp": end_timestamp,
    "dur_mean": moving_avg_duration,
    "dur_std": moving_std_duration,
    "ts_mean": moving_avg_ts,
    "ts_std": moving_std_ts,
})

# Save
window_df.to_csv("Dataset/windowedSet.csv", index=False)
print(window_df.head())

                   start_timestamp                    end_timestamp  dur_mean  \
0 2025-10-31 22:55:39.633569+00:00 2025-10-31 22:56:34.866330+00:00  0.006682   
1 2025-10-31 22:55:47.097758+00:00 2025-10-31 22:56:43.481462+00:00  0.006835   
2 2025-10-31 22:55:55.704890+00:00 2025-10-31 22:56:44.085920+00:00  0.006798   
3 2025-10-31 22:55:57.455345+00:00 2025-10-31 22:56:49.837661+00:00  0.006792   
4 2025-10-31 22:56:02.635641+00:00 2025-10-31 22:56:55.019935+00:00  0.006831   

    dur_std   ts_mean    ts_std  
0  0.000696  1.104655  1.130780  
1  0.000620  1.139244  0.986436  
2  0.000662  0.990660  0.775908  
3  0.000661  1.059236  0.924353  
4  0.000739  1.059255  0.972643  
Empty DataFrame
Columns: [start_timestamp, end_timestamp, dur_mean, dur_std, ts_mean, ts_std]
Index: []
Empty DataFrame
Columns: [start_timestamp, end_timestamp, dur_mean, dur_std, ts_mean, ts_std]
Index: []
